# 1 · NASA C-MAPSS data preparation

Start here. The demo reads the saved audit, split and preprocessing metadata and a prepared FD001 test example. It works on a Colab CPU without Drive.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display

repo_url = "https://github.com/abhav-arora/ML3103-project.git"
repo = next((p for p in (Path.cwd(), Path.cwd().parent)
             if (p / "spikedistill").is_dir() and (p / "results").is_dir()), None)
if repo is None:
    repo = Path("/content/ML3103-demo")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo)], check=True)
print("Project source ready.")

Project source ready.


## Data audit and split

Each row is one engine cycle. An engine belongs to a single fit or validation group. Official test engines are scored separately, once at their last observed cycle.

In [2]:
audit_root = repo / "results/data_audit"
summary = json.loads((audit_root / "audit_summary.json").read_text())
display(pd.DataFrame([{
    "subset": row["subset"], "official_train_engines": row["official_train"]["engines"],
    "official_test_engines": row["official_test"]["engines"], "fit_engines": row["training_engines"],
    "validation_engines": row["validation_engines"], "feature_count": row["features"],
    "training_windows": row["arrays"]["train"]["shape"][0]
} for row in summary]))

manifest = json.loads((audit_root / "FD001/split_manifest.json").read_text())
assert set(manifest["train_engine_ids"]).isdisjoint(manifest["validation_engine_ids"])
print("Fixed split seed:", manifest["seed"], "| window:", manifest["window"], "| cap:", manifest["rul_cap"])
print(manifest["validation_protocol"])

Fixed split seed: 42 | window: 30 | cap: 125.0
one prefix per held-out engine; fraction U(0.4,0.9), seed+1000


,subset,official_train_engines,official_test_engines,fit_engines,validation_engines,feature_count,training_windows
0,FD001,100,100,80,20,18,14459
1,FD002,260,259,208,52,24,37051
2,FD003,100,100,80,20,19,17754
3,FD004,249,248,199,50,24,42960


## Train-only normalization and input example

We remove near-constant sensors on the training engines and fit the feature scaler there. FD002 and FD004 use six training-fitted operating regimes. The training target is capped at 125 cycles and scaled by 125. The main official test comparison uses raw RUL in cycles.

In [3]:
import numpy as np

prep = json.loads((audit_root / "FD001/preprocessing.json").read_text())
print("FD001 feature order:", ", ".join(prep["feature_names"]))
print("Dropped sensors:", ", ".join(prep["dropped_sensors"]))
fixture = repo / "deployment/jetson/onnx/test_vectors.npz"
with np.load(fixture, allow_pickle=False) as test:
    print("One input window shape:", test["sensor_window"][0].shape)
    print("Official test windows in this sample file:", len(test["engine_id"]))
    print("Observed cycles in the extra padding check:", int(test["valid_mask"][-1].sum()))
    display(pd.DataFrame(test["sensor_window"][0, -5:], columns=prep["feature_names"]).round(2))

FD001 feature order: sensor_02, sensor_03, sensor_04, sensor_06, sensor_07, sensor_08, sensor_09, sensor_11, sensor_12, sensor_13, sensor_14, sensor_15, sensor_17, sensor_20, sensor_21, op_1, op_2, op_3
Dropped sensors: sensor_01, sensor_05, sensor_10, sensor_16, sensor_18, sensor_19
One input window shape: (30, 18)
Official test windows in this sample file: 100
Observed cycles in the extra padding check: 11


,sensor_02,sensor_03,sensor_04,sensor_06,sensor_07,sensor_08,sensor_09,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,op_1,op_2,op_3
0,-1.20,-0.63,-0.96,0.14,1.11,-0.10,-0.89,-0.75,0.55,-1.07,-0.89,0.19,-0.78,0.30,0.95,-0.31,0.33,0.0
1,-1.50,0.61,-0.85,0.14,0.22,-0.38,-0.44,-1.84,0.58,-0.37,-0.47,0.13,-0.78,0.08,0.56,1.02,1.70,0.0
2,-1.46,-0.55,-1.21,0.14,0.88,-0.24,-0.89,-0.45,1.33,-0.37,-0.56,-0.56,-0.78,1.13,0.67,0.65,0.33,0.0
3,0.22,-0.79,-0.89,0.14,0.83,-0.10,-0.83,-0.53,0.50,0.05,-0.47,-0.83,-1.43,1.52,1.08,-1.13,1.35,0.0
4,-0.20,-1.52,-1.12,0.14,1.19,-0.24,-0.40,-1.16,0.51,-0.51,-0.73,-1.06,-0.14,-0.03,0.60,-0.26,1.35,0.0


## Rebuild when needed

Run this command from the GitHub checkout with the NASA text files available. The original Baselines notebook performs this preparation before training.

    python -m spikedistill.data.prepare --data-dir /path/to/CMaps --output-dir artifacts/data

Next open [Baselines.ipynb](https://colab.research.google.com/github/abhav-arora/ML3103-project/blob/main/notebooks/Baselines.ipynb).